# R.O.M. decoder test on S0-2 radial velocities

This notebook tests the decoding of β, e, ω_i and i from the radial-velocity stream alone, using the S0-2 readings in `DATA/S0-2_DataS1_full.csv`.

The decoding follows these steps:

1. **Period T.** The readings from the earlier pass (2000–2002) are shifted forward until they land on the curve traced by the later pass (2018).
2. **β.** All readings are placed inside one period, and the average of Λ_raw is taken over that period on the star's own clock. Each arrival time is first moved back by the light-travel delay, which is the running time integral of the line-of-sight speed. Then τ = T / ∫Λ_raw dt, and β = √(3 − √(1 + 8τ²)) / 2.
3. **Λ_A and Λ_D.** These are the highest and lowest readings, together with their arrival times.
4. **e and ω_i.** These come from the Holographic Decryption Invariant (amplitude) and the time from maximum to minimum (timing), solved by Newton iteration with your Jacobian.
5. **i.** This is i = arcsin((D_A − D_D) √(1 − e²) / (2β)), reported as folded and unfolded (180° − i). The same equations are also solved at the phases where the model's raw shift peaks, with i as a third unknown.

In [ ]:
# =====================================================================
# Tests the radial-velocity-only decoding of beta, e, omega_i and i
# on the S0-2 data, step by step, and on perfect synthetic readings
# taken at the same dates (to separate method error from data error).
# =====================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from scipy.integrate import quad, trapezoid, cumulative_trapezoid
from scipy.optimize import least_squares, minimize_scalar

# ---------------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------------
DATA_URL = "https://raw.githubusercontent.com/AntonRize/WILL/refs/heads/main/DATA/S0-2_DataS1_full.csv"
C_KMS = 299792.458            # speed of light, km/s
GM_SUN = 1.32712440018e20     # m^3/s^2
YEAR_S = 365.25 * 86400.0     # Julian year, s

# NIRC2 instrument offset fitted by Do et al. 2019 (Science 365, 664), Table 1:
# +81 km/s (statistical uncertainty 19 km/s). NIRC2 readings in the file sit
# about 100 km/s below the other instruments at neighbouring dates.
NIRC2_OFFSET_KMS = 81.0

N_REPEATS = 200               # repeats with readings shifted within their errors
RNG_SEED = 1

# Published S0-2 orbit (Do et al. 2019, Table 1, medians), from the same
# radial velocities plus astrometry. Used only for comparison and for the
# synthetic check.
PUB = dict(P=16.042, T0=2018.3763, e=0.8858, omega=66.11, i=133.82, M_sun=3.975e6)


# ---------------------------------------------------------------------
# R.O.M. RELATIONS (from the R.O.M. equation set)
# ---------------------------------------------------------------------
def z_sys(O, beta, e):
    """System shift Z_sys(O) = 1/tau_o(O) at true phase O."""
    eY2 = 1.0 - e**2
    kappa_o2 = 2.0 * beta**2 * (1.0 + e * np.cos(O)) / eY2
    beta_o2 = beta**2 * (1.0 + e**2 + 2.0 * e * np.cos(O)) / eY2
    return 1.0 / np.sqrt((1.0 - kappa_o2) * (1.0 - beta_o2))

def z_raw(O, beta, e, w, inc):
    """Raw shift Z_raw(O) = (1 + beta_z(O)) Z_sys(O)."""
    beta_z = beta / np.sqrt(1.0 - e**2) * (np.cos(O + w) + e * np.cos(w)) * np.sin(inc)
    return (1.0 + beta_z) * z_sys(O, beta, e)

def tau_node(beta, e, w, sign):
    """tau at the ascending node (sign=+1) or descending node (sign=-1)."""
    c = sign * np.cos(w)
    u = (1.0 + e**2 + 2.0 * e * c) / (1.0 - e**2)
    v = (1.0 + e * c) / (1.0 - e**2)
    return np.sqrt(1.0 - beta**2 * u) * np.sqrt(1.0 - 2.0 * beta**2 * v), u, v

def beta_from_tau(tau):
    """beta = sqrt(3 - sqrt(1 + 8 tau^2)) / 2, real only for tau < 1."""
    if not np.isfinite(tau) or tau >= 1.0:
        return np.nan
    return np.sqrt(3.0 - np.sqrt(1.0 + 8.0 * tau**2)) / 2.0

def zeta_unwrapped(O, e):
    """zeta(O) = (1-e^2)^(3/2) * integral_0^O (1+e cos)^-2, in closed form,
    continuous in O across whole turns."""
    O = np.asarray(O, float)
    O0 = np.mod(O + np.pi, 2 * np.pi) - np.pi
    k = np.round((O - O0) / (2 * np.pi))
    E = 2 * np.arctan2(np.sqrt(1 - e) * np.sin(O0 / 2), np.sqrt(1 + e) * np.cos(O0 / 2))
    return E - e * np.sin(E) + 2 * np.pi * k

def elapsed_zeta(Oa, Ob, e):
    """zeta(Ob) - zeta(Oa): elapsed phase time between true phases Oa and Ob."""
    return zeta_unwrapped(Ob, e) - zeta_unwrapped(Oa, e)

def dzeta_de_elapsed(Oa, Ob, e):
    pts = [0.0] if Oa < 0.0 < Ob else None
    i1, _ = quad(lambda th: 1.0 / (1.0 + e * np.cos(th))**2, Oa, Ob, points=pts, limit=200)
    i2, _ = quad(lambda th: np.cos(th) / (1.0 + e * np.cos(th))**3, Oa, Ob, points=pts, limit=200)
    return -3.0 * e * np.sqrt(1.0 - e**2) * i1 - 2.0 * (1.0 - e**2)**1.5 * i2


# ---------------------------------------------------------------------
# DATA HANDLING
# ---------------------------------------------------------------------
def mjd_to_year(mjd):
    return 2000.0 + (np.asarray(mjd, float) - 51544.5) / 365.25

def load_readings(url, nirc2_offset):
    df = pd.read_csv(url)
    df.columns = df.columns.str.strip()
    t = mjd_to_year(df["MJD"])
    v = df["RV_km_s"].to_numpy(float).copy()
    s = df["sigma_km_s"].to_numpy(float)
    ins = df["Instrument"].astype(str).str.strip().to_numpy()
    v[ins == "NIRC2"] += nirc2_offset
    return t, v, s, ins

def merge_close(t, v, s, tol_yr=1.0 / 365.25):
    """Combine readings taken less than one day apart (error-weighted)."""
    o = np.argsort(t)
    t, v, s = t[o], v[o], s[o]
    ot, ov, os_ = [], [], []
    i = 0
    while i < len(t):
        j = i
        while j + 1 < len(t) and t[j + 1] - t[i] < tol_yr:
            j += 1
        w = 1.0 / s[i:j + 1]**2
        ot.append(np.sum(w * t[i:j + 1]) / w.sum())
        ov.append(np.sum(w * v[i:j + 1]) / w.sum())
        os_.append(1.0 / np.sqrt(w.sum()))
        i = j + 1
    return np.array(ot), np.array(ov), np.array(os_)


# ---------------------------------------------------------------------
# STEP 1: PERIOD T FROM THE TWO PASSES
# ---------------------------------------------------------------------
def measure_period(t, v, s, T_lo=15.8, T_hi=16.2):
    """Shift the readings of the earlier pass by T and find the T at which
    they land on the curve traced by the later pass."""
    t_end = t.max()
    early = t < t_end - T_hi          # earlier pass: lands inside the later pass for every trial T
    tl, vl, _ = merge_close(t[~early], v[~early], s[~early])
    curve = PchipInterpolator(tl, vl, extrapolate=False)
    grid = np.arange(T_lo, T_hi, 2e-5)
    model = curve(t[early][:, None] + grid[None, :])
    chi2 = np.sum(((v[early][:, None] - model) / s[early][:, None])**2, axis=0)
    k = np.nanargmin(chi2)
    inside = grid[chi2 <= chi2[k] + 1.0]
    return grid[k], inside.min(), inside.max()


# ---------------------------------------------------------------------
# STEP 2: ONE-PERIOD CURVE AND ITS AVERAGE ON THE STAR'S CLOCK
# ---------------------------------------------------------------------
def one_period_curve(t, v, s, T):
    """Place every reading inside one period window [t_end - T, t_end]."""
    t_end = t.max()
    t_start = t_end - T
    k = np.maximum(np.ceil((t_start - t) / T), 0.0)
    tf, vf, sf = merge_close(t + k * T, v, s)
    te = np.concatenate([tf - T, tf, tf + T])
    curve = PchipInterpolator(te, np.tile(vf, 3))
    return curve, tf, vf, sf, t_start

def average_ratio(curve, t_start, T, n=200001):
    """Average of Lambda_raw = 1 + v/c over one period, on the arrival clock
    and on the star's own clock (arrival time minus light-travel delay)."""
    g = np.linspace(t_start, t_start + T, n)
    vg = curve(g)
    mean_arrival = trapezoid(vg, g) / T
    # line-of-sight displacement relative to its one-period average position
    z = cumulative_trapezoid(vg - mean_arrival, g, initial=0.0)   # km/s * yr
    t_star = g - z / C_KMS                                        # yr
    mean_star = trapezoid(vg, t_star) / (t_star[-1] - t_star[0])
    delay = lambda tt: np.interp(t_start + np.mod(tt - t_start, T), g, z) / C_KMS
    return 1.0 + mean_arrival / C_KMS, 1.0 + mean_star / C_KMS, delay


# ---------------------------------------------------------------------
# STEP 3: MAXIMUM AND MINIMUM READINGS AND THEIR TIMES
# ---------------------------------------------------------------------
def extreme(tf, vf, kind):
    """Highest (kind='max') or lowest (kind='min') reading and its arrival time,
    exactly as the method prescribes: no curve fitting."""
    k = np.argmax(vf) if kind == "max" else np.argmin(vf)
    return tf[k], vf[k]


# ---------------------------------------------------------------------
# STEP 4: e AND omega_i FROM AMPLITUDE AND TIMING (Newton, your Jacobian)
# ---------------------------------------------------------------------
def F_and_J(e, w, beta, LA, LD, frac):
    c, s = np.cos(w), np.sin(w)
    tA, uA, vA = tau_node(beta, e, w, +1)
    tD, uD, vD = tau_node(beta, e, w, -1)
    F1 = LA * tA * (1 - e * c) + LD * tD * (1 + e * c) - 2.0
    F2 = elapsed_zeta(-w, np.pi - w, e) - 2.0 * np.pi * frac

    def dtau(t_, u, v, du, dv):
        return t_ * (-beta**2 * du / (2 * (1 - beta**2 * u)) - beta**2 * dv / (1 - 2 * beta**2 * v))
    den = (1 - e**2)**2
    duA_e, dvA_e = (4 * e + 2 * (1 + e**2) * c) / den, (2 * e + (1 + e**2) * c) / den
    duD_e, dvD_e = (4 * e - 2 * (1 + e**2) * c) / den, (2 * e - (1 + e**2) * c) / den
    duA_w, dvA_w = -2 * e * s / (1 - e**2), -e * s / (1 - e**2)
    J11 = LA * (dtau(tA, uA, vA, duA_e, dvA_e) * (1 - e * c) - tA * c) + LD * (dtau(tD, uD, vD, duD_e, dvD_e) * (1 + e * c) + tD * c)
    J12 = LA * (dtau(tA, uA, vA, duA_w, dvA_w) * (1 - e * c) + tA * e * s) + LD * (dtau(tD, uD, vD, -duA_w, -dvA_w) * (1 + e * c) - tD * e * s)
    J21 = dzeta_de_elapsed(-w, np.pi - w, e)
    J22 = -4 * e * c * (1 - e**2)**1.5 / (1 - e**2 * c**2)**2
    return np.array([F1, F2]), np.array([[J11, J12], [J21, J22]])

def solve_e_omega(beta, LA, LD, frac):
    b = 0.0 if not np.isfinite(beta) else beta
    E, W = np.meshgrid(np.linspace(0.02, 0.98, 97), np.radians(np.arange(0.0, 360.0, 2.0)), indexing="ij")
    C = np.cos(W)
    tA = tau_node(b, E, W, +1)[0]
    tD = tau_node(b, E, W, -1)[0]
    r1 = (LA * tA * (1 - E * C) + LD * tD * (1 + E * C) - 2.0) / (LA - LD)
    r2 = (elapsed_zeta(-W, np.pi - W, E) - 2 * np.pi * frac) / (2 * np.pi)
    k = np.unravel_index(np.argmin(r1**2 + r2**2), E.shape)
    X = np.array([E[k], W[k]])
    for _ in range(50):
        F, J = F_and_J(X[0], X[1], b, LA, LD, frac)
        dX = np.linalg.solve(J, F)
        step = 1.0
        while not (0.0 < X[0] - step * dX[0] < 1.0) and step > 1e-6:
            step *= 0.5
        X = X - step * dX
        if np.max(np.abs(dX)) < 1e-13:
            break
    return X[0], np.mod(X[1], 2 * np.pi)

def inclination(beta, e, w, LA, LD):
    tA = tau_node(beta, e, w, +1)[0]
    tD = tau_node(beta, e, w, -1)[0]
    sin_i = (LA * tA - LD * tD) / (2.0 * beta) * np.sqrt(1.0 - e**2)
    return np.degrees(np.arcsin(sin_i)) if 0.0 < sin_i <= 1.0 else np.nan


# ---------------------------------------------------------------------
# STEP 5: CORRECTED SOLVE AT THE PHASES OF THE OBSERVED EXTREMES
# ---------------------------------------------------------------------
def extreme_phase(beta, e, w, inc, O_node, kind):
    sgn = -1.0 if kind == "max" else 1.0
    r = minimize_scalar(lambda O: sgn * z_raw(O, beta, e, w, inc),
                        bounds=(O_node - 0.3, O_node + 0.3), method="bounded",
                        options={"xatol": 1e-12})
    return r.x

def solve_at_extremes(beta, LA, LD, frac, e0, w0, i0_deg):
    def resid(p):
        e, w, inc = p
        OA = extreme_phase(beta, e, w, inc, -w, "max")
        OD = extreme_phase(beta, e, w, inc, np.pi - w, "min")
        return [(z_raw(OA, beta, e, w, inc) - LA) / (LA - LD),
                (z_raw(OD, beta, e, w, inc) - LD) / (LA - LD),
                (elapsed_zeta(OA, OD, e) - 2 * np.pi * frac) / (2 * np.pi)]
    r = least_squares(resid, [e0, w0, np.radians(i0_deg)],
                      bounds=([0.01, w0 - 0.5, 0.01], [0.99, w0 + 0.5, np.pi / 2]),
                      xtol=1e-14, ftol=1e-14, gtol=1e-14)
    e, w, inc = r.x
    return e, np.mod(w, 2 * np.pi), np.degrees(inc)


# ---------------------------------------------------------------------
# FULL PIPELINE
# ---------------------------------------------------------------------
def decode(t, v, s, corrected=True):
    out = {}
    T, T_lo, T_hi = measure_period(t, v, s)
    curve, tf, vf, sf, t_start = one_period_curve(t, v, s, T)
    L_arr, L_star, delay = average_ratio(curve, t_start, T)
    tA, vA = extreme(tf, vf, "max")
    tD, vD = extreme(tf, vf, "min")
    if tD < tA:
        tD += T
    dt_star = (tD - tA) - (delay(tD) - delay(tA))
    LA, LD = 1.0 + vA / C_KMS, 1.0 + vD / C_KMS
    tau = 1.0 / L_star
    beta = beta_from_tau(tau)
    beta_arr = beta_from_tau(1.0 / L_arr)
    e, w = solve_e_omega(beta, LA, LD, dt_star / T)
    i = inclination(beta, e, w, LA, LD) if np.isfinite(beta) else np.nan
    out.update(T=T, T_lo=T_lo, T_hi=T_hi, mean_arr=(L_arr - 1) * C_KMS, mean_star=(L_star - 1) * C_KMS,
               beta=beta, beta_arr=beta_arr, tA=tA, vA=vA, tD=tD, vD=vD,
               dt=dt_star, e=e, w=np.degrees(w), i=i, curve=curve, tf=tf, vf=vf, t_start=t_start)
    if corrected and np.isfinite(beta) and np.isfinite(i):
        e2, w2, i2 = solve_at_extremes(beta, LA, LD, dt_star / T, e, w, i)
        out.update(e_c=e2, w_c=np.degrees(w2), i_c=i2)
    else:
        out.update(e_c=np.nan, w_c=np.nan, i_c=np.nan)
    return out


# ---------------------------------------------------------------------
# SYNTHETIC READINGS FROM THE R.O.M. MODEL AT THE SAME DATES
# ---------------------------------------------------------------------
def true_phase(t_emit, P, T0, e):
    M = np.mod(2 * np.pi * (t_emit - T0) / P + np.pi, 2 * np.pi) - np.pi
    E = M.copy()
    for _ in range(100):
        E -= (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
    return 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2), np.sqrt(1 - e) * np.cos(E / 2))

def synthetic_rv(t_arrival, beta, P, T0, e, w_deg, i_deg):
    w, inc = np.radians(w_deg), np.radians(i_deg)
    a_light_yr = P * beta / (2 * np.pi)               # a/c in years, from a = T beta c / (2 pi)
    t_emit = t_arrival.copy()
    for _ in range(30):
        O = true_phase(t_emit, P, T0, e)
        r = a_light_yr * (1 - e**2) / (1 + e * np.cos(O))
        t_emit = t_arrival - r * np.sin(O + w) * np.sin(inc)   # light-travel delay
    O = true_phase(t_emit, P, T0, e)
    return C_KMS * (z_raw(O, beta, e, w, inc) - 1.0)


# ---------------------------------------------------------------------
# RUN
# ---------------------------------------------------------------------
def report(tag, r):
    print(f"\n=== {tag} ===")
    print(f"Orbital period T (time for the earlier pass to land on the later pass): {r['T']:.4f} yr "
          f"(readings agree within their errors for {r['T_lo']:.4f} to {r['T_hi']:.4f} yr)")
    print(f"One-period average of the readings, arrival clock:   {r['mean_arr']:+.2f} km/s")
    print(f"One-period average of the readings, star's clock:    {r['mean_star']:+.2f} km/s")
    print(f"beta from the star's-clock average (tau = T / integral of Lambda_raw dt): "
          + (f"{r['beta']:.6f}" if np.isfinite(r['beta']) else "none (average is not redder than rest)"))
    print(f"beta if the arrival clock were used instead:                              "
          + (f"{r['beta_arr']:.6f}" if np.isfinite(r['beta_arr']) else "none"))
    print(f"Highest reading (Lambda_A): {r['vA']:+.1f} km/s at {r['tA']:.4f}")
    print(f"Lowest reading  (Lambda_D): {r['vD']:+.1f} km/s at {r['tD']:.4f}")
    print(f"Time from maximum to minimum on the star's clock: {r['dt']:.4f} yr")
    print(f"e        (amplitude + timing, node equations): {r['e']:.4f}")
    print(f"omega_i  (amplitude + timing, node equations): {r['w']:.2f} deg")
    if np.isfinite(r['i']):
        print(f"i        (arcsin, folded / unfolded):          {r['i']:.2f} / {180 - r['i']:.2f} deg")
    else:
        print("i        : no solution (sin i would exceed 1 or beta is undefined)")
    if np.isfinite(r['e_c']):
        print(f"Equations written at the extreme phases: e = {r['e_c']:.4f}, omega_i = {r['w_c']:.2f} deg, "
              f"i = {r['i_c']:.2f} / {180 - r['i_c']:.2f} deg")


## What each run shows

- **A0** uses perfect synthetic readings from the R.O.M. model every 2 days, with the published S0-2 orbit as input. It checks the method itself.
- **A** uses perfect synthetic readings at the real observation dates. It shows what the gaps in this data set cost.
- **B** uses the real readings exactly as they are in the file.
- **C** uses the real readings with the NIRC2 readings shifted by the instrument offset that Do et al. fitted (+81 km/s).
- **D** repeats C 200 times, with every reading shifted at random within its stated error. It shows how far the measurement errors alone move each result.
- The last lines give the error-weighted mean of the readings minus the published-orbit model, both away from pericentre and during the 2018 passage.

In [ ]:
beta_pub = (np.pi * 2 * GM_SUN * PUB["M_sun"] / (C_KMS * 1e3)**2 / (PUB["P"] * YEAR_S * C_KMS * 1e3))**(1 / 3)
print("Published S0-2 orbit (Do et al. 2019, Science 365, 664, Table 1):")
print(f"  P = {PUB['P']} yr, e = {PUB['e']}, omega = {PUB['omega']} deg, i = {PUB['i']} deg")
print(f"  beta implied by the published mass {PUB['M_sun']:.3e} M_sun and period: {beta_pub:.6f}")
print(f"  expected one-period average of the readings, c(1/tau - 1): "
      f"{(1 / np.sqrt((1 - 2 * beta_pub**2) * (1 - beta_pub**2)) - 1) * C_KMS:.2f} km/s")

t, v_file, s, ins = load_readings(DATA_URL, 0.0)
_, v_corr, _, _ = load_readings(DATA_URL, NIRC2_OFFSET_KMS)

# A0. Perfect synthetic readings every 2 days (no gaps): checks the method itself
t_dense = np.arange(2000.0, 2018.75, 2.0 / 365.25)
s_dense = np.full_like(t_dense, 15.0)
v_dense = synthetic_rv(t_dense, beta_pub, PUB["P"], PUB["T0"], PUB["e"], PUB["omega"], PUB["i"])
report("A0. Perfect synthetic readings every 2 days (method check)", decode(t_dense, v_dense, s_dense))

# A. Perfect synthetic readings on the real dates
v_syn = synthetic_rv(t, beta_pub, PUB["P"], PUB["T0"], PUB["e"], PUB["omega"], PUB["i"])
r_syn = decode(t, v_syn, s)
report("A. Perfect synthetic readings at the real observation dates", r_syn)

# B. Real readings, as in the file
r_file = decode(t, v_file, s)
report("B. Real readings as in the file", r_file)

# C. Real readings with the NIRC2 offset applied
r_corr = decode(t, v_corr, s)
report(f"C. Real readings, NIRC2 shifted by +{NIRC2_OFFSET_KMS:.0f} km/s", r_corr)

# Level of the real readings relative to the published orbit (C readings)
resid = v_corr - v_syn
print("\nReading minus published-orbit model (error-weighted mean):")
for lo, hi, label in [(2000.0, 2018.2, "away from pericentre, 2000 to 2018.2"),
                      (2018.2, 2019.0, "2018 pericentre passage")]:
    k = (t >= lo) & (t < hi)
    print(f"  {label:38s} {np.sum(resid[k] / s[k]**2) / np.sum(1 / s[k]**2):+7.2f} km/s  ({k.sum()} readings)")

# D. Repeat C with every reading shifted at random within its stated error
rng = np.random.default_rng(RNG_SEED)
keys = ["T", "mean_star", "beta", "e", "w", "i"]
rows = []
for n in range(N_REPEATS):
    rr = decode(t, v_corr + rng.normal(0.0, s), s, corrected=False)
    rows.append([rr[k] for k in keys])
rows = np.array(rows, float)
print(f"\n=== D. Spread over {N_REPEATS} repeats with readings shifted within their errors (C) ===")
names = ["T [yr]", "average, star's clock [km/s]", "beta", "e", "omega_i [deg]", "i folded [deg]"]
for name, col in zip(names, rows.T):
    ok = np.isfinite(col)
    if ok.sum() > 2:
        lo, mid, hi = np.percentile(col[ok], [16, 50, 84])
        print(f"  {name:32s} middle {mid:10.5f}, 68% of repeats between {lo:.5f} and {hi:.5f} "
              f"({ok.sum()} of {N_REPEATS} repeats give a value)")
    else:
        print(f"  {name:32s} no value in {N_REPEATS - ok.sum()} of {N_REPEATS} repeats")

# Figure: readings in one period window and the reconstructed curve
r = r_corr
T = r["T"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, (x0, x1) in zip(ax, [(r["t_start"], r["t_start"] + T), (2018.1, 2019.4)]):
    gg = np.linspace(x0, x1, 20000)
    a.plot(gg, r["curve"](gg), lw=1, label="curve through the readings")
    a.plot(r["tf"], r["vf"], "o", ms=3, label="readings (one period window)")
    a.plot(r["tf"] + T, r["vf"], "o", ms=3, color="C1")
    a.plot([r["tA"], r["tD"]], [r["vA"], r["vD"]], "rx", ms=10, mew=2, label="max / min")
    a.set_xlim(x0, x1); a.set_xlabel("arrival time [yr]"); a.set_ylabel("radial velocity [km/s]")
ax[0].legend(fontsize=8)
ax[1].set_title("pericentre passage and minimum")
plt.tight_layout()
plt.show()
